# ResNet-50 From Scratch for Pascal VOC 2012

This notebook trains a **multi-label** image classifier from scratch on the local Pascal VOC 2012 dataset.
Pascal VOC images can contain more than one object class, so this notebook uses multi-label targets and `BCEWithLogitsLoss` instead of `CrossEntropyLoss`.

## 1. Imports

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import torch
import torch.nn as nn
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


## 2. Device Setup

In [ ]:
# Main settings you may want to change.
data_dir = Path('VOCdevkit') / 'VOC2012'
train_split = 'train'
val_split = 'val'
image_size = 224
batch_size = 16
num_epochs = 8
learning_rate = 1e-4
weight_decay = 1e-4
random_seed = 42
num_workers = 0
update_every = 100
prediction_threshold = 0.5
best_model_path = 'resnet50_voc_best.pth'
final_model_path = 'resnet50_voc_final.pth'

# Pascal VOC has 20 object classes.
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Standard normalization values for RGB images.
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 3. Dataset Loading

In [ ]:
class VOCDataset(Dataset):
    def __init__(self, root_path, split_name, transform=None):
        root_path = Path(root_path)
        self.split_name = split_name
        self.transform = transform
        image_dir = root_path / 'JPEGImages'
        self.annotation_dir = root_path / 'Annotations'
        split_file = root_path / 'ImageSets' / 'Main' / f'{split_name}.txt'

        required_paths = [root_path, image_dir, self.annotation_dir, split_file]
        missing_paths = [str(path) for path in required_paths if not path.exists()]
        if missing_paths:
            raise FileNotFoundError('Missing VOC paths:\n' + '\n'.join(missing_paths))

        class_to_index = {name: index for index, name in enumerate(VOC_CLASSES)}
        self.samples = []

        for image_id in split_file.read_text().splitlines():
            image_id = image_id.strip()
            if not image_id:
                continue

            image_path = image_dir / f'{image_id}.jpg'
            annotation_path = self.annotation_dir / f'{image_id}.xml'

            if not image_path.exists() or not annotation_path.exists():
                continue

            target = torch.zeros(len(VOC_CLASSES), dtype=torch.float32)
            for obj in ET.parse(annotation_path).getroot().findall('object'):
                class_name = obj.findtext('name')
                if class_name in class_to_index:
                    target[class_to_index[class_name]] = 1.0

            self.samples.append((image_id, image_path, target))

        if not self.samples:
            raise ValueError(f'No valid image/annotation pairs were found for split: {split_name}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        _, image_path, target = self.samples[index]
        image = Image.open(image_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, target.clone()


class_names = VOC_CLASSES
num_classes = len(class_names)

print(f'Dataset root: {data_dir}')
print(f'Number of classes: {num_classes}')


## 4. Data Transforms

In [ ]:
# Keep the transforms simple and readable.
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print('Transforms are ready.')


## 5. DataLoader

In [ ]:
train_dataset = VOCDataset(root_path=data_dir, split_name=train_split, transform=train_transform)
val_dataset = VOCDataset(root_path=data_dir, split_name=val_split, transform=val_transform)

pin_memory = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Train batches per epoch: {len(train_loader)}')
print(f'Validation batches per epoch: {len(val_loader)}')


## 6. Model Definition

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, bottleneck_channels, stride=1, downsample=None):
        super().__init__()
        out_channels = bottleneck_channels * self.expansion

        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)

        self.conv2 = nn.Conv2d(
            bottleneck_channels,
            bottleneck_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)

        self.conv3 = nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes):
        super().__init__()
        self.in_channels = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(block, bottleneck_channels=64, blocks=layers[0], stride=1)
        self.layer2 = self._make_layer(block, bottleneck_channels=128, blocks=layers[1], stride=2)
        self.layer3 = self._make_layer(block, bottleneck_channels=256, blocks=layers[2], stride=2)
        self.layer4 = self._make_layer(block, bottleneck_channels=512, blocks=layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        self._initialize_weights()

    def _make_layer(self, block, bottleneck_channels, blocks, stride):
        downsample = None
        out_channels = bottleneck_channels * block.expansion

        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = [block(self.in_channels, bottleneck_channels, stride=stride, downsample=downsample)]
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, bottleneck_channels))

        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def resnet50(num_classes):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)


## 7. Loss and Optimizer

In [ ]:
model = resnet50(num_classes=num_classes).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print(f'Number of classes: {num_classes}')
print(f'Total parameters: {total_parameters:,}')
print(f'Trainable parameters: {trainable_parameters:,}')

# Quick smoke test so we know the model output shape is correct.
with torch.no_grad():
    dummy_inputs = torch.randn(2, 3, image_size, image_size, device=device)
    dummy_outputs = model(dummy_inputs)

print(f'Smoke test output shape: {tuple(dummy_outputs.shape)}')


## 8. Accuracy Calculation

In [ ]:
def calculate_multilabel_accuracy(logits, targets, threshold=0.5):
    '''Measure how many class decisions are correct across the whole batch.'''
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()
    accuracy = (predictions == targets).float().mean().item()
    return accuracy


def create_checkpoint(model, optimizer, epoch, best_val_accuracy, train_losses, train_accuracies, val_losses, val_accuracies):
    '''Store the most important training information in one dictionary.'''
    return {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'class_names': class_names,
        'num_classes': num_classes,
        'data_dir': str(data_dir),
        'train_split': train_split,
        'val_split': val_split,
        'image_size': image_size,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'prediction_threshold': prediction_threshold,
        'best_val_accuracy': best_val_accuracy,
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'task_type': 'pascal_voc_multilabel_classification',
    }


## 9. Training Loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch_number, total_epochs, update_every, threshold):
    model.train()
    running_loss = 0.0
    running_accuracy = 0.0
    running_total = 0
    average_loss = 0.0
    average_accuracy = 0.0

    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch_number}/{total_epochs} - Train', leave=False)

    for step, (images, targets) in enumerate(progress_bar, start=1):
        images = images.to(device, non_blocking=device.type == 'cuda')
        targets = targets.to(device, non_blocking=device.type == 'cuda')

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        batch_size_now = targets.size(0)
        batch_accuracy = calculate_multilabel_accuracy(outputs, targets, threshold=threshold)

        running_loss += loss.item() * batch_size_now
        running_accuracy += batch_accuracy * batch_size_now
        running_total += batch_size_now

        average_loss = running_loss / running_total
        average_accuracy = running_accuracy / running_total

        progress_bar.set_postfix(loss=f'{average_loss:.4f}', acc=f'{average_accuracy * 100:.2f}%')

        if step % update_every == 0 or step == len(dataloader):
            tqdm.write(
                f'Epoch {epoch_number}/{total_epochs} | '
                f'Step {step}/{len(dataloader)} | '
                f'Train Loss: {average_loss:.4f} | '
                f'Train Accuracy: {average_accuracy * 100:.2f}%'
            )

    return average_loss, average_accuracy


## 10. Validation Loop

In [ ]:
def validate_one_epoch(model, dataloader, criterion, device, epoch_number, total_epochs, threshold):
    model.eval()
    running_loss = 0.0
    running_accuracy = 0.0
    running_total = 0
    average_loss = 0.0
    average_accuracy = 0.0

    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch_number}/{total_epochs} - Val', leave=False)

    with torch.no_grad():
        for images, targets in progress_bar:
            images = images.to(device, non_blocking=device.type == 'cuda')
            targets = targets.to(device, non_blocking=device.type == 'cuda')

            outputs = model(images)
            loss = criterion(outputs, targets)

            batch_size_now = targets.size(0)
            batch_accuracy = calculate_multilabel_accuracy(outputs, targets, threshold=threshold)

            running_loss += loss.item() * batch_size_now
            running_accuracy += batch_accuracy * batch_size_now
            running_total += batch_size_now

            average_loss = running_loss / running_total
            average_accuracy = running_accuracy / running_total

            progress_bar.set_postfix(loss=f'{average_loss:.4f}', acc=f'{average_accuracy * 100:.2f}%')

    return average_loss, average_accuracy


## 11. Train the Model

In [ ]:
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

best_val_accuracy = -1.0

for epoch in range(1, num_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epoch_number=epoch,
        total_epochs=num_epochs,
        update_every=update_every,
        threshold=prediction_threshold,
    )

    val_loss, val_accuracy = validate_one_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
        epoch_number=epoch,
        total_epochs=num_epochs,
        threshold=prediction_threshold,
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(
        f'Epoch {epoch}/{num_epochs} Summary | '
        f'Train Loss: {train_loss:.4f} | '
        f'Train Accuracy: {train_accuracy * 100:.2f}% | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Accuracy: {val_accuracy * 100:.2f}%'
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_checkpoint = create_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_val_accuracy=best_val_accuracy,
            train_losses=train_losses,
            train_accuracies=train_accuracies,
            val_losses=val_losses,
            val_accuracies=val_accuracies,
        )
        torch.save(best_checkpoint, best_model_path)
        print(f'Saved a new best model to {best_model_path} (val accuracy: {val_accuracy * 100:.2f}%)')

print('Training finished.')


## 12. Model Saving

In [ ]:
final_checkpoint = create_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=num_epochs,
    best_val_accuracy=best_val_accuracy,
    train_losses=train_losses,
    train_accuracies=train_accuracies,
    val_losses=val_losses,
    val_accuracies=val_accuracies,
)

torch.save(final_checkpoint, final_model_path)

print(f'Best model saved to: {best_model_path}')
print(f'Final model saved to: {final_model_path}')
print(f'Best validation accuracy: {best_val_accuracy * 100:.2f}%')


## 13. Visualisation

In [ ]:
epoch_numbers = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epoch_numbers, train_losses, marker='o', label='Train Loss')
plt.plot(epoch_numbers, val_losses, marker='o', label='Validation Loss')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_numbers, [score * 100 for score in train_accuracies], marker='o', label='Train Accuracy')
plt.plot(epoch_numbers, [score * 100 for score in val_accuracies], marker='o', label='Validation Accuracy')
plt.title('Multi-Label Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

## 14. Random object identification

In [ ]:


def load_best_model_for_visualization(checkpoint_path, num_classes, device):
    '''Load the best saved checkpoint into a fresh ResNet-50 model.''' 
    checkpoint_file = Path(checkpoint_path)
    if not checkpoint_file.exists():
        raise FileNotFoundError(
            f"Best model checkpoint '{checkpoint_file}' was not found. Run the training cells first."
        )

    checkpoint = torch.load(checkpoint_file, map_location=device)
    best_model = resnet50(num_classes=num_classes).to(device)
    best_model.load_state_dict(checkpoint['model_state_dict'])
    best_model.eval()
    return best_model


def parse_voc_objects_for_visualization(annotation_path):
    '''Read VOC object names and boxes without depending on earlier notebook helpers.'''
    import xml.etree.ElementTree as ET

    xml_root = ET.parse(annotation_path).getroot()
    objects = []

    for obj in xml_root.findall('object'):
        class_name = obj.findtext('name')
        bbox = obj.find('bndbox')
        if class_name is None or bbox is None:
            continue

        objects.append({
            'name': class_name,
            'xmin': int(float(bbox.findtext('xmin', 0))),
            'ymin': int(float(bbox.findtext('ymin', 0))),
            'xmax': int(float(bbox.findtext('xmax', 0))),
            'ymax': int(float(bbox.findtext('ymax', 0))),
        })

    return objects


def show_random_voc_sample_with_best_model(dataset, class_names, best_model, transform, device, threshold=0.5, seed=None):
    '''Show one random image with VOC boxes and semantic predictions from the best model.'''
    if seed is not None:
        random.seed(seed)

    sample_index = random.randrange(len(dataset.samples))
    sample = dataset.samples[sample_index]
    if len(sample) == 4:
        image_id, image_path, annotation_path, target = sample
    elif len(sample) == 3:
        image_id, image_path, target = sample
        annotation_path = dataset.annotation_dir / f'{image_id}.xml'
    else:
        raise ValueError(
            f'Unexpected dataset sample format with {len(sample)} values. '
            'Expected either 3 or 4 values.'
        )

    objects = parse_voc_objects_for_visualization(annotation_path)
    image = Image.open(image_path).convert('RGB')

    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = best_model(input_tensor)
        probabilities = torch.sigmoid(logits)[0].cpu()

    predicted_indices = (probabilities >= threshold).nonzero(as_tuple=True)[0].tolist()
    if predicted_indices:
        predicted_labels = [
            f"{class_names[index]} ({probabilities[index]:.2f})"
            for index in predicted_indices
        ]
    else:
        top_count = min(3, len(class_names))
        top_scores, top_indices = torch.topk(probabilities, k=top_count)
        predicted_labels = [
            f"{class_names[index.item()]} ({score.item():.2f})"
            for score, index in zip(top_scores, top_indices)
        ]

    ground_truth_labels = sorted(set(obj['name'] for obj in objects))
    ground_truth_summary = ', '.join(ground_truth_labels) if ground_truth_labels else 'No labels found'
    prediction_summary = ', '.join(predicted_labels)

    figure, axis = plt.subplots(figsize=(12, 9))
    axis.imshow(image)

    color_map = plt.get_cmap('tab20', len(class_names))
    for obj in objects:
        class_index = class_names.index(obj['name'])
        color = color_map(class_index)
        width = obj['xmax'] - obj['xmin']
        height = obj['ymax'] - obj['ymin']

        rectangle = Rectangle(
            (obj['xmin'], obj['ymin']),
            width,
            height,
            linewidth=2,
            edgecolor=color,
            facecolor='none',
        )
        axis.add_patch(rectangle)

        axis.text(
            obj['xmin'],
            max(obj['ymin'] - 5, 5),
            obj['name'],
            color='white',
            fontsize=10,
            bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=2),
        )

    figure.suptitle(f'Best-model view on random {dataset.split_name} image: {image_id}', fontsize=14)
    figure.text(
        0.5,
        0.03,
        f'Ground truth: {ground_truth_summary}\nPredicted: {prediction_summary}',
        ha='center',
        va='bottom',
        fontsize=10,
        bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=6),
    )
    axis.axis('off')
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.show()

    print(f'Image id: {image_id}')
    print(f'Objects found: {len(objects)}')
    print('Ground-truth labels:', ground_truth_summary)
    print('Predicted labels from best model:', prediction_summary)
    print('Note: rectangles come from VOC annotations, while the semantic predictions come from the best classifier checkpoint.')

    return image_id, ground_truth_labels, predicted_labels


best_visual_model = load_best_model_for_visualization(best_model_path, num_classes, device)
show_random_voc_sample_with_best_model(
    dataset=val_dataset,
    class_names=class_names,
    best_model=best_visual_model,
    transform=val_transform,
    device=device,
    threshold=prediction_threshold,
)